# Day 6 阈值成本敏感性分析

Day 6 不重新训练模型，只读取 Day 4 / Day 5 已生成的预测概率文件，对不同阈值下的 precision、recall、F1、F2、FP、FN 和 total cost 做回溯敏感性分析。

特别注意：当前阈值结果基于测试集预测概率做回溯分析，不能写成生产环境最终阈值。更严谨的流程应使用验证集选择阈值，再在测试集上评估。

## 1. 读取 cfg 和依赖

成本参数和输出目录继续从 `config/config.yaml` 读取，不在 notebook 中写死 FP/FN 成本。

In [2]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from scania_aps.config import get_config
from scania_aps.evaluation.threshold_utils import build_threshold_summary, evaluate_threshold_grid

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

cfg = get_config(PROJECT_ROOT / "config" / "config.yaml")
cfg.false_positive_cost, cfg.false_negative_cost, cfg.default_threshold

(10, 500, 0.5)

## 2. 读取 Day 4 / Day 5 predictions

这里只读取已有预测结果，不调用训练脚本。

In [3]:
prediction_sources = [
    (cfg.predictions_dir / "day4_baseline_predictions.csv", "day4_baseline"),
    (cfg.predictions_dir / "day5_model_compare_predictions.csv", "day5_advanced"),
]

prediction_parts = []
for path, source in prediction_sources:
    temp = pd.read_csv(path)
    temp["source"] = source
    prediction_parts.append(temp)

predictions = pd.concat(prediction_parts, ignore_index=True)
predictions.groupby(["source", "model_name", "strategy"]).size().reset_index(name="rows")

,source,model_name,strategy,rows
0,day4_baseline,dummy_prior,drop_high_missing_median,16000
1,day4_baseline,dummy_prior,median_all,16000
2,day4_baseline,logistic_regression_balanced,drop_high_missing_median,16000
3,day4_baseline,logistic_regression_balanced,median_all,16000
4,day5_advanced,random_forest_balanced,drop_high_missing_median,16000
5,day5_advanced,random_forest_balanced,median_all,16000
6,day5_advanced,random_forest_balanced,median_with_indicator,16000
7,day5_advanced,xgboost_scale_pos_weight,drop_high_missing_median,16000
8,day5_advanced,xgboost_scale_pos_weight,median_all,16000


## 3. 默认阈值结果回顾

默认阈值 `0.5` 并不一定适合 APS 成本场景。FN 成本远高于 FP 时，业务上通常更关注漏报数量和 total cost，而不是默认分类阈值。

In [4]:
default_metrics = pd.concat(
    [
        pd.read_csv(cfg.metrics_dir / "day4_baseline_metrics.csv"),
        pd.read_csv(cfg.metrics_dir / "day5_model_compare_metrics.csv"),
    ],
    ignore_index=True,
)

default_view = default_metrics[
    ["model_name", "strategy", "threshold", "precision", "recall", "f2", "average_precision", "fp", "fn", "total_cost"]
].sort_values("total_cost")

default_view

,model_name,strategy,threshold,precision,recall,f2,average_precision,fp,fn,total_cost
3,logistic_regression_balanced,drop_high_missing_median,0.5,0.486034,0.928000,0.785199,0.799356,368,27,17180
7,xgboost_scale_pos_weight,drop_high_missing_median,0.5,0.627737,0.917333,0.839844,0.909098,204,31,17540
1,logistic_regression_balanced,median_all,0.5,0.481894,0.922667,0.779982,0.798196,372,29,18220
5,xgboost_scale_pos_weight,median_all,0.5,0.635688,0.912000,0.839058,0.911301,196,33,18460
4,random_forest_balanced,median_all,0.5,0.936709,0.592000,0.639033,0.884151,15,153,76650
8,random_forest_balanced,median_with_indicator,0.5,0.948498,0.589333,0.637623,0.890959,12,154,77120
6,random_forest_balanced,drop_high_missing_median,0.5,0.938326,0.568000,0.616676,0.883036,14,162,81140
2,dummy_prior,drop_high_missing_median,0.5,0.000000,0.000000,0.000000,0.023438,0,375,187500
0,dummy_prior,median_all,0.5,0.000000,0.000000,0.000000,0.023438,0,375,187500


## 4. 运行 threshold grid

阈值从 0.01 到 0.99，步长 0.01。Dummy baseline 虽然有概率列，但只是对照；后续解释重点放在 Logistic 和 XGBoost。

In [5]:
threshold_parts = []
candidate_predictions = predictions[predictions["y_proba"].notna()].copy()

for (model_name, strategy), group in candidate_predictions.groupby(["model_name", "strategy"], dropna=False):
    result = evaluate_threshold_grid(
        y_true=group["y_true"],
        y_proba=group["y_proba"],
        cfg=cfg,
        model_name=model_name,
        strategy=strategy,
    )
    threshold_parts.append(result)

threshold_metrics = pd.concat(threshold_parts, ignore_index=True)
best_summary = build_threshold_summary(threshold_metrics)

threshold_metrics.shape, best_summary

((891, 15),
                      model_name                  strategy  best_threshold  precision    recall        f1        f2     fp  fn  total_cost  \
 8      xgboost_scale_pos_weight                median_all            0.20   0.469231  0.976000  0.633766  0.802632    414   9        8640   
 5        random_forest_balanced                median_all            0.03   0.393583  0.981333  0.561832  0.755647    567   7        9170   
 4        random_forest_balanced  drop_high_missing_median            0.02   0.340367  0.989333  0.506485  0.716216    719   4        9190   
 7      xgboost_scale_pos_weight  drop_high_missing_median            0.08   0.358252  0.984000  0.525267  0.729249    661   6        9610   
 6        random_forest_balanced     median_with_indicator            0.03   0.391258  0.978667  0.559025  0.752666    571   8        9710   
 2  logistic_regression_balanced  drop_high_missing_median            0.30   0.395787  0.952000  0.559123  0.743131    545  18       144

## 5. 默认阈值 vs 低成本阈值

这里比较默认阈值 0.5 和当前测试集回溯分析中的低成本阈值。阈值降低通常会提高 recall、减少 FN，但也会增加 FP，因此必须看 total cost。

In [6]:
default_best_compare = default_metrics.merge(
    best_summary,
    on=["model_name", "strategy"],
    how="inner",
    suffixes=("_default", "_best"),
)

default_best_compare = default_best_compare[
    [
        "model_name",
        "strategy",
        "threshold",
        "best_threshold",
        "recall_default",
        "recall_best",
        "precision_default",
        "precision_best",
        "f2_default",
        "f2_best",
        "fn_default",
        "fn_best",
        "fp_default",
        "fp_best",
        "total_cost_default",
        "total_cost_best",
    ]
].sort_values("total_cost_best")

default_best_compare

,model_name,strategy,threshold,best_threshold,recall_default,recall_best,precision_default,precision_best,f2_default,f2_best,fn_default,fn_best,fp_default,fp_best,total_cost_default,total_cost_best
5,xgboost_scale_pos_weight,median_all,0.5,0.20,0.912000,0.976000,0.635688,0.469231,0.839058,0.802632,33,9,196,414,18460,8640
4,random_forest_balanced,median_all,0.5,0.03,0.592000,0.981333,0.936709,0.393583,0.639033,0.755647,153,7,15,567,76650,9170
6,random_forest_balanced,drop_high_missing_median,0.5,0.02,0.568000,0.989333,0.938326,0.340367,0.616676,0.716216,162,4,14,719,81140,9190
7,xgboost_scale_pos_weight,drop_high_missing_median,0.5,0.08,0.917333,0.984000,0.627737,0.358252,0.839844,0.729249,31,6,204,661,17540,9610
8,random_forest_balanced,median_with_indicator,0.5,0.03,0.589333,0.978667,0.948498,0.391258,0.637623,0.752666,154,8,12,571,77120,9710
3,logistic_regression_balanced,drop_high_missing_median,0.5,0.30,0.928000,0.952000,0.486034,0.395787,0.785199,0.743131,27,18,368,545,17180,14450
1,logistic_regression_balanced,median_all,0.5,0.25,0.922667,0.954667,0.481894,0.363821,0.779982,0.720612,29,17,372,626,18220,14760
2,dummy_prior,drop_high_missing_median,0.5,0.01,0.000000,1.000000,0.000000,0.023438,0.000000,0.107143,375,0,0,15625,187500,156250
0,dummy_prior,median_all,0.5,0.01,0.000000,1.000000,0.000000,0.023438,0.000000,0.107143,375,0,0,15625,187500,156250


## 6. 保存 Day 6 输出表

结果表保存到 `outputs/metrics/`。这些是本地分析产物，默认不提交 GitHub。

In [7]:
cfg.metrics_dir.mkdir(parents=True, exist_ok=True)

threshold_metrics_path = cfg.metrics_dir / "day6_threshold_metrics.csv"
best_summary_path = cfg.metrics_dir / "day6_best_threshold_summary.csv"

threshold_metrics.to_csv(threshold_metrics_path, index=False, encoding="utf-8-sig")
best_summary.to_csv(best_summary_path, index=False, encoding="utf-8-sig")

threshold_metrics_path, best_summary_path

(WindowsPath('C:/Scania APS/outputs/metrics/day6_threshold_metrics.csv'),
 WindowsPath('C:/Scania APS/outputs/metrics/day6_best_threshold_summary.csv'))

## 7. 绘制阈值曲线

图表只展示重点模型，避免曲线过密：Logistic + `drop_high_missing_median`、XGBoost + `drop_high_missing_median`、XGBoost + `median_all`。

In [8]:
focus_pairs = [
    ("logistic_regression_balanced", "drop_high_missing_median"),
    ("xgboost_scale_pos_weight", "drop_high_missing_median"),
    ("xgboost_scale_pos_weight", "median_all"),
]

plot_parts = []
for model_name, strategy in focus_pairs:
    part = threshold_metrics[
        threshold_metrics["model_name"].eq(model_name)
        & threshold_metrics["strategy"].eq(strategy)
    ]
    if not part.empty:
        plot_parts.append(part)

plot_data = pd.concat(plot_parts, ignore_index=True)
plot_data[["model_name", "strategy"]].drop_duplicates()

,model_name,strategy
0,logistic_regression_balanced,drop_high_missing_median
99,xgboost_scale_pos_weight,drop_high_missing_median
198,xgboost_scale_pos_weight,median_all


In [9]:
cfg.figures_dir.mkdir(parents=True, exist_ok=True)

def series_label(row):
    return f"{row['model_name']} | {row['strategy']}"

plt.figure(figsize=(10, 6))
for _, group in plot_data.groupby(["model_name", "strategy"]):
    plt.plot(group["threshold"], group["total_cost"], label=series_label(group.iloc[0]))
plt.title("Day 6 threshold vs total cost")
plt.xlabel("Threshold")
plt.ylabel("Total cost")
plt.legend()
plt.tight_layout()
plt.savefig(cfg.figures_dir / "day6_threshold_cost_curve.png", dpi=160)
plt.close()

plt.figure(figsize=(10, 6))
for _, group in plot_data.groupby(["model_name", "strategy"]):
    label = series_label(group.iloc[0])
    plt.plot(group["threshold"], group["precision"], linestyle="--", label=f"{label} precision")
    plt.plot(group["threshold"], group["recall"], linestyle="-", label=f"{label} recall")
plt.title("Day 6 threshold vs precision / recall")
plt.xlabel("Threshold")
plt.ylabel("Metric value")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(cfg.figures_dir / "day6_threshold_precision_recall_curve.png", dpi=160)
plt.close()

plt.figure(figsize=(10, 6))
for _, group in plot_data.groupby(["model_name", "strategy"]):
    plt.plot(group["threshold"], group["f2"], label=series_label(group.iloc[0]))
plt.title("Day 6 threshold vs F2")
plt.xlabel("Threshold")
plt.ylabel("F2")
plt.legend()
plt.tight_layout()
plt.savefig(cfg.figures_dir / "day6_threshold_f2_curve.png", dpi=160)
plt.close()

[
    cfg.figures_dir / "day6_threshold_cost_curve.png",
    cfg.figures_dir / "day6_threshold_precision_recall_curve.png",
    cfg.figures_dir / "day6_threshold_f2_curve.png",
]

[WindowsPath('C:/Scania APS/outputs/figures/day6_threshold_cost_curve.png'),
 WindowsPath('C:/Scania APS/outputs/figures/day6_threshold_precision_recall_curve.png'),
 WindowsPath('C:/Scania APS/outputs/figures/day6_threshold_f2_curve.png')]

## 8. Day 6 小结

- 默认 0.5 阈值并不一定适合 APS 成本场景，因为 FN 成本远高于 FP。
- 阈值降低会倾向于提高 recall、减少 FN，但会增加 FP，因此必须结合 total cost 判断。
- 不能只看 PR-AUC 或 F2：PR-AUC 反映排序能力，F2 偏向 recall，但最终维修决策仍需要看业务成本。
- 当前结果是测试集回溯敏感性分析，不是生产环境最终阈值。
- 当前值得进入 Day 7 风险分层准备的是 XGBoost + `median_all`，但 Day 7 仍应把阈值结论表述为当前项目阶段的分析建议。